In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

In [ ]:
def draw_and_compare(true_slope, true_int, sample_size):
    x = np.random.normal(50, 5, sample_size)
    xlims = np.array([np.min(x), np.max(x)])
    errors = np.random.normal(0, 6, sample_size)
    y = (true_slope * x + true_int) + errors
    sample = Table().with_columns('x', x, 'y', y)

    sample.scatter('x', 'y')
    plots.plot(xlims, true_slope*xlims + true_int, lw=2, color='green')
    plots.title('True Line, and Points Created')

    sample.scatter('x', 'y')
    plots.title('What We Get to See')

    sample.scatter('x', 'y', fit_line=True)
    plots.title('Regression Line: Estimate of True Line')

    sample.scatter('x', 'y', fit_line=True)
    plots.plot(xlims, true_slope*xlims + true_int, lw=2, color='green')
    plots.title("Regression Line and True Line")

## Linear regression

In [ ]:
def standard_units(x):
    """Converts an array x to standard units"""
    return (x - np.mean(x)) / np.std(x)

def correlation(t, x, y):
    x_su = standard_units(t.column(x))
    y_su = standard_units(t.column(y))
    return np.mean(x_su * y_su)

def slope(t, x, y):
    """t is a table; x and y are column labels"""
    r = correlation(t, x, y)
    slope_xy = r * np.std(t.column(y)) / np.std(t.column(x))
    return slope_xy

def intercept(t, x, y):
    """t is a table; x and y are column labels"""
    slope_xy = slope(t, x, y)
    intercept_xy = np.average(t.column(y) - slope_xy * np.average(t.column(x)))
    return intercept_xy

def fitted_values(t, x, y):
    """Return the height of the regression line at each x value"""
    a = slope(t, x, y)
    b = intercept(t, x, y)
    return a * t.column(x) + b

def residual(t, x, y):
    """Return the residuals of the regression line at each x value."""
    return t.column(y) - fitted_values(t, x, y)

def residual_plot(t, x, y):
    """displays a residual plot for the least squares regression line"""
    Table().with_columns(
            'fitted values', fitted_values(t, x, y),
            'residuals', residual(t, x, y)
    ).scatter('fitted values', 'residuals', color='r')
    plots.axhline(0, color='darkblue', lw=4)
    plots.title('Residual Plot')

def prediction_at(t, x, y, x_value):
    """ return the least squares regression prediction for x_value"""
    return slope(t, x, y) * x_value + intercept(t, x, y)

## Regression Model  - Sample vs Population Lines ##

In [ ]:
draw_and_compare(-2, 5, 50)

## Prediction ##

In [ ]:
births = Table.read_table('/home/jovyan/shared/Data/baby.csv')
births = births.where('Gestational Days', are.between(240, 310))
births.show(3)

In [ ]:
births.scatter('Gestational Days', 'Birth Weight', fit_line=True)

In [ ]:
prediction_at_280 = prediction_at(births, 'Gestational Days', 'Birth Weight', 280)
prediction_at_280

## Confidence Interval for Prediction ##

In [ ]:
def bootstrap_prediction(t, x, y, new_x, repetitions=1000):

    # Bootstrap the scatter, predict, collect
    predictions = make_array()
    for i in np.arange(repetitions):
        resample = t.sample()
        predicted_y = prediction_at(resample, x, y, new_x)
        predictions = np.append(predictions, predicted_y)

    # Find the ends of the approximate 95% prediction interval
    left = percentile(2.5, predictions)
    right = percentile(97.5, predictions)

    # Display results
    Table().with_column('Prediction', predictions).hist(bins=20)
    plots.xlabel('predictions at x='+str(new_x))
    plots.plot([left, right], [0, 0], color='yellow', lw=8);
    print('Approximate 95%-confidence interval for height of true line:')
    print(left, right, '(width =', right - left, ')') 

In [ ]:
bootstrap_prediction(births, 'Gestational Days', 'Birth Weight', 280)

## Inference for the Slope ##

In [ ]:
births.scatter('Gestational Days', 'Birth Weight', fit_line=True)

In [ ]:
slope(births, 'Gestational Days', 'Birth Weight')

In [ ]:
def bootstrap_slope(t, x, y, repetitions=1000):
    
    # Bootstrap the scatter, find the slope, collect
    slopes = make_array()
    for i in np.arange(repetitions):
        resample = t.sample()
        bootstrap_slope = slope(resample, x, y)
        slopes = np.append(slopes, bootstrap_slope)
    
    # Find the endpoints of the 95% confidence interval for the true slope
    left = percentile(2.5, slopes)
    right = percentile(97.5, slopes)
    
    # Slope of the regression line from the original sample
    observed_slope = slope(t, x, y)
    
    # Display results
    Table().with_column('Bootstrap Slopes', slopes).hist(bins=20)
    plots.plot(make_array(left, right), make_array(0, 0), color='yellow', lw=8);
    print('Slope of regression line:', observed_slope)
    print('Approximate 95%-confidence interval for the slope of the true line:')
    print(left, 'to', right)

In [ ]:
bootstrap_slope(births, 'Gestational Days', 'Birth Weight')

## Rain on the Regression Parade

In [ ]:
draw_and_compare(0, 10, 25)

**Null Hypothesis.** Slope of true line = 0.

**Alternative Hypothesis.** Slope of true line is not 0.

In [ ]:
slope(births, 'Maternal Age', 'Birth Weight')

In [ ]:
births.scatter('Maternal Age', 'Birth Weight', fit_line=True)

In [ ]:
bootstrap_slope(births, 'Maternal Age', 'Birth Weight')

## Back to HighPeaks

In [ ]:
HighPeaks = Table.read_table('/home/jovyan/shared/Data/HighPeaks.csv')
HighPeaks.scatter('Ascent', 'Time', fit_line = True)
slope(HighPeaks, 'Ascent', 'Time')

In [ ]:
residual_plot(HighPeaks, 'Ascent', 'Time')

In [ ]:
bootstrap_slope(HighPeaks, 'Ascent', 'Time', repetitions = 10000)

## Multiple Regression

In [ ]:
births.scatter('Gestational Days', 'Birth Weight', fit_line = True)
correlation(births, 'Birth Weight', 'Gestational Days')

In [ ]:
births.scatter('Maternal Age', 'Birth Weight', fit_line = True)
correlation(births, 'Birth Weight', 'Maternal Age')

In [ ]:
births.scatter('Maternal Height', 'Birth Weight', fit_line = True)
correlation(births, 'Birth Weight', 'Maternal Height')

In [ ]:
births.scatter('Maternal Pregnancy Weight', 'Birth Weight', fit_line = True)
correlation(births, 'Birth Weight', 'Maternal Pregnancy Weight')

What if we try to predict birth weight from gestational days, maternal height, and maternal pregnancy weight?

In [ ]:
def multiple_regression_rmse(gest_days_mult, mat_height_mult, mat_preg_weight_mult, intercept):
    
    gest_days = births.column('Gestational Days')
    mat_height = births.column('Maternal Height')
    mat_preg_weight = births.column('Maternal Pregnancy Weight')
    
    y = births.column('Birth Weight')
    
    prediction = sum([
        gest_days_mult * gest_days,
        mat_height_mult * mat_height,
        mat_preg_weight_mult * mat_preg_weight,
        intercept]
    )
    
    mse = np.mean((y - prediction) ** 2)
    return np.sqrt(mse)

In [ ]:
best = minimize(multiple_regression_rmse)
best

This means that our prediction for birth weight is:

0.3899927(*Gestational Days*) + 0.06350791(*Maternal Height*) + 0.1065575(*Maternal Pregnancy Weight*) + -6.46133206

In [ ]:
# Prediction for birthweight for 280 gestational days, Maternal Height 65", Maternal Pregnancy Weight 165

best.item(0) * 280 + best.item(1) * 65 + best.item(2) * 165 + best.item(3)

## Residual Analysis

In [ ]:
mult_predictions = sum([
    best.item(0) * births.column('Gestational Days'),
    best.item(1) * births.column('Maternal Height'),
    best.item(2) * births.column('Maternal Pregnancy Weight'),
    best.item(3)]
)

mult_resid = births.column('Birth Weight') - mult_predictions 

In [ ]:
births = births.with_columns(
    'Mult Predictions', mult_predictions,
    'Mult Residuals', mult_resid 
)

In [ ]:
births.scatter('Mult Predictions', 'Mult Residuals', color = 'red')
plots.axhline(0, color = 'darkblue', lw = 4);

## Confidence Interval for Multiple Regression Prediction

In [ ]:
def bootstrap_prediction_3pred(t, w, x, z, y, new_w, new_x, new_z, repetitions=1000):

    # Bootstrap the scatter, predict, collect
    predictions = make_array()
    for i in np.arange(repetitions):
        resample = t.sample()
        
        def multiple_regression_rmse(a, b, c, d):
    
            prediction = a * resample.column(w) + b * resample.column(x) + c * resample.column(z) + d
    
            mse = np.mean((resample.column(y) - prediction) ** 2)
            return np.sqrt(mse)
        
        coef = minimize(multiple_regression_rmse)
        predicted_y = coef.item(0) * new_w + coef.item(1) * new_x + coef.item(2) * new_z + coef.item(3)
        predictions = np.append(predictions, predicted_y)

    # Find the ends of the approximate 95% prediction interval
    left = percentile(2.5, predictions)
    right = percentile(97.5, predictions)

    # Display results
    Table().with_column('Prediction', predictions).hist(bins=15)
    plots.plot([left, right], [0, 0], color='yellow', lw=8);
    print('Approximate 95%-confidence interval for height of true line:')
    print(left, right, '(width =', right - left, ')') 

In [ ]:
bootstrap_prediction_3pred(births, 
                           'Gestational Days', 'Maternal Height', 'Maternal Pregnancy Weight', 'Birth Weight', 
                           280, 65, 165)